# Explorador e Assistente do DATASUS 🇧🇷🏥

Este Jupyter Notebook é um caderno de estudos e uma ferramenta interativa para consultar, baixar e analisar dados públicos do **DATASUS** (Departamento de Informática do SUS).

Ele foi estruturado especificamente para apoiar a elaboração de **trabalhos científicos, TCCs e artigos**, automatizando o download da base de dados e a geração de visualizações.

### Principais Sistemas de Informação Disponíveis:
*   **SIM (Sistema de Informações sobre Mortalidade)**: Dados de óbitos, contendo causas de óbito (CID-10), idade, sexo, escolaridade, local de ocorrência, etc.
*   **SINASC (Sistema de Informações sobre Nascidos Vivos)**: Dados de nascimentos, contendo peso ao nascer, tipo de parto, consultas pré-natal, idade da mãe, etc.
*   **SIHSUS (Sistema de Informações Hospitalares)**: Internações hospitalares financiadas pelo SUS (diagnóstico, custo, tempo de permanência, etc.).
*   **SIASUS (Sistema de Informações Ambulatoriais)**: Procedimentos ambulatoriais realizados.
*   **CNES (Cadastro Nacional de Estabelecimentos de Saúde)**: Dados de leitos, infraestrutura e profissionais de saúde cadastrados.

## 1. Importação de Bibliotecas e Módulo Core

Importamos as bibliotecas de visualização e carregamos as funções do módulo **`datasus_core.py`**, que centraliza o download do FTP, a descompressão local e o cache Parquet.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importa as funções principais do arquivo de núcleo (datasus_core.py)
from datasus_core import buscar_dados, baixar_arquivo_datasus, CACHE_DIR

# Configurações estéticas para os gráficos do artigo científico
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print(f"✅ Módulo datasus_core importado com sucesso!")
print(f"📂 Pasta local de cache: {CACHE_DIR}")

## 2. Teste de Busca em Ação

Escreva uma pergunta sobre o tema que deseja estudar na variável `sua_pergunta` abaixo e execute a célula. O sistema fará a interpretação em linguagem natural e o download automático!

In [ ]:
# Exemplos de perguntas:
# - "Quero analisar os nascimentos no Acre em 2022"
# - "Buscar os óbitos ocorridos em Roraima no ano de 2022"
# - "Visualizar as internações hospitalares no Acre em janeiro de 2022"

sua_pergunta = "Quero analisar os nascimentos no Acre em 2022"

# Executa a busca inteligente
df = buscar_dados(sua_pergunta)

# Mostra as primeiras 5 linhas da tabela baixada
df.head()

## 3. Visualização e Análise dos Resultados

Abaixo, rodamos uma análise exploratória automática baseada no tipo de dados que foram carregados na memória.

In [ ]:
print(f"📊 Tamanho do Banco de Dados: {df.shape[0]:,} linhas por {df.shape[1]} variáveis.")

In [ ]:
# Executa um script de análise padrão de acordo com o tipo de dados retornado

if 'IDADEMAE' in df.columns:
    print("\n--- 📊 ANÁLISE DE NASCIMENTOS (SINASC) ---")
    
    # Converter Idade da Mãe para número
    df['IDADEMAE'] = pd.to_numeric(df['IDADEMAE'], errors='coerce')
    
    # Histograma de Idade da Mãe
    plt.figure(figsize=(10, 5))
    sns.histplot(data=df, x='IDADEMAE', kde=True, color='teal', bins=25)
    plt.title("Distribuição da Idade das Mães", fontsize=14, fontweight='bold')
    plt.xlabel("Idade da Mãe (Anos)")
    plt.ylabel("Frequência")
    plt.show()
    
    # Distribuição por tipo de parto
    if 'PARTO' in df.columns:
        parto_labels = {'1': 'Vaginal', '2': 'Cesáreo'}
        df['Tipo_Parto'] = df['PARTO'].astype(str).map(parto_labels).fillna('Outros/Ignorado')
        
        plt.figure(figsize=(7, 4))
        sns.countplot(data=df, x='Tipo_Parto', palette='Set2', hue='Tipo_Parto', legend=False)
        plt.title("Tipo de Parto Registrado", fontsize=14, fontweight='bold')
        plt.xlabel("Tipo de Parto")
        plt.ylabel("Frequência")
        plt.show()
        
elif 'CAUSABAS' in df.columns:
    print("\n--- 📊 ANÁLISE DE MORTALIDADE (SIM) ---")
    
    # Top 10 causas de morte
    top10_morte = df['CAUSABAS'].value_counts().head(10).reset_index()
    top10_morte.columns = ['CID-10', 'Óbitos']
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top10_morte, x='Óbitos', y='CID-10', palette='Reds_r', hue='CID-10', legend=False)
    plt.title("Top 10 Causas de Óbito (CIDs mais frequentes)", fontsize=14, fontweight='bold')
    plt.xlabel("Quantidade de Óbitos")
    plt.ylabel("Causa Básica (Código CID-10)")
    plt.show()
    
    # Distribuição por Sexo (SEXO: 1=Masculino, 2=Feminino, 0=Ignorado)
    if 'SEXO' in df.columns:
        sexo_labels = {'1': 'Masculino', '2': 'Feminino', '0': 'Ignorado'}
        df['Sexo_Label'] = df['SEXO'].astype(str).map(sexo_labels).fillna('Não informado')
        
        plt.figure(figsize=(7, 4))
        sns.countplot(data=df, x='Sexo_Label', palette='coolwarm', hue='Sexo_Label', legend=False)
        plt.title("Óbitos por Sexo", fontsize=14, fontweight='bold')
        plt.xlabel("Sexo")
        plt.ylabel("Óbitos")
        plt.show()

elif 'DIAG_PRINC' in df.columns:
    print("\n--- 📊 ANÁLISE DE INTERNAÇÕES HOSPITALARES (SIH) ---")
    
    # Diagnósticos principais de internação mais comuns
    top_internacoes = df['DIAG_PRINC'].value_counts().head(10).reset_index()
    top_internacoes.columns = ['Diagnóstico (CID)', 'Internações']
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=top_internacoes, x='Internações', y='Diagnóstico (CID)', palette='viridis', hue='Diagnóstico (CID)', legend=False)
    plt.title("Top 10 Causas Principais de Internação Hospitalar", fontsize=14, fontweight='bold')
    plt.xlabel("Quantidade de Internações")
    plt.ylabel("Causa de Internação (Código CID)")
    plt.show()
    
else:
    print("\n📊 O banco de dados foi carregado e está pronto. Use os métodos como 'df.describe()' ou 'df.info()' para explorar outras colunas.")